# Level 3 Scoring Analysis — 2025–26 Season

## Analytical Objective

Determine which scoring components differentiate successful Level 3 All Star
cheerleading performances while separating routine construction from performance
errors and competition-specific scoring formats.

## Analytical Grain

The primary observation is one team performance within a competition, division,
and round.

Primary outcomes:
- Performance score
- Placement within competition / division / round

Event score is retained as source data but is not treated as a universally
comparable outcome because event aggregation rules vary by competition.

## Analysis Lenses

1. **All performances**
   - Measures actual competitive outcomes, including deductions and missed skills.

2. **Zero-deduction performances**
   - Separates performances with no recorded deductions to reduce the influence
     of falls and other penalized performance errors.
   - Zero deductions does not necessarily indicate that every intended or
     compulsory skill was successfully performed. Omitted or uncredited skills
     may reduce category difficulty scores without producing a deduction.

3. **Deduction analysis**
   - Examines deductions separately to quantify the competitive impact of
     performance errors.

## Scoring Context

Many Level 3 difficulty categories are effectively compulsory: teams know the
skills required to maximize these categories and generally construct routines
to meet those requirements.

These include:
- Stunt difficulty
- Stunt DOD
- Stunt MAX / MPD
- Toss difficulty
- Standing tumbling difficulty
- Standing tumbling DOD
- Running tumbling difficulty
- Running tumbling DOD
- Running tumbling MAX / MPD
- Jump difficulty

A missed or dropped skill can simultaneously produce a deduction and cause the
skill to receive reduced or no difficulty credit. Therefore, difficulty scores
from performances with deductions may reflect execution failure rather than
routine-construction decisions.

Difficulty-category relationships will therefore be evaluated separately for
all performances and hit performances.

Pyramid difficulty and dance difficulty differ from the compulsory difficulty
categories above because their difficulty scores are subjectively evaluated.
They will therefore be analyzed separately from the objective/max-out
difficulty categories.

This distinction is important when interpreting variance. Low variance in a
compulsory difficulty category among clean performances does not necessarily
mean the category is unimportant; it may indicate that competitive teams
routinely construct routines to maximize the available score.

### Category Scale Interpretation

Raw category point ranges are not directly comparable measures of judging
variability. Execution categories use technical scoring ranges in which scores
begin at a maximum and are reduced in similar increments for identified
execution issues, despite categories having different nominal maximum values.

Accordingly, category importance will not be inferred from raw score range or
percentage of the nominal maximum alone. Later comparative models will use
within-category standardization where appropriate while preserving the
different scoring mechanisms represented by each rubric.

## Competition Format Considerations

Competition scoring formats are not uniform.

- Many two-day competitions use approximately 25% preliminary / 75% final
  weighting.
- CHEERSPORT weights the team's higher-scoring day at 75%.
- Summit advancement rounds do not use score carryover.

For this reason, individual round performance is the primary analytical unit.
Competition-specific event aggregation will be handled separately when needed.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_PATH = Path(
    "../data/processed/season_2026_level3_performances.csv"
)

df = pd.read_csv(DATA_PATH)

print(f"Performances: {len(df):,}")
print(f"Competitions: {df['competition_id'].nunique():,}")
print(f"Teams: {df['team_id'].nunique():,}")
print(f"Divisions: {df['division_id'].nunique():,}")
print()
print("Rounds:")
print(df["round"].value_counts())

Performances: 6,517
Competitions: 157
Teams: 884
Divisions: 58

Rounds:
round
Finals        2734
Semifinals    1240
Prelims       1132
Round 1        749
Round 2        430
Wild Card      232
Name: count, dtype: int64


In [3]:
analysis_df = df.copy()

analysis_df["is_hit"] = (
    analysis_df["deductions"] == 0
)

hit_summary = (
    analysis_df["is_hit"]
    .value_counts()
    .rename(index={
        True: "Hit / zero deductions",
        False: "Deduction"
    })
    .to_frame("performances")
)

hit_summary["pct"] = (
    hit_summary["performances"]
    / len(analysis_df)
    * 100
)

print(hit_summary)

print("\nDeduction amounts:")
print(
    analysis_df["deductions"]
    .value_counts()
    .sort_index()
    .to_string()
)

                       performances     pct
is_hit                                     
Hit / zero deductions          3369 51.6956
Deduction                      3148 48.3044

Deduction amounts:
deductions
0.0000    3369
0.0100       1
0.0500      12
0.1000     122
0.1500     206
0.2000      11
0.2500     748
0.3000      26
0.3500      43
0.4000      70
0.4500       7
0.5000     143
0.5500       8
0.6000      14
0.6500      19
0.7000       4
0.7500     534
0.8000       7
0.8500      33
0.9000      63
0.9500       8
1.0000     201
1.0500       4
1.1000      18
1.1500      23
1.2000       5
1.2500     187
1.3000       5
1.3500      10
1.4000      23
1.4500       5
1.5000     149
1.5500       1
1.6000      12
1.6500      27
1.7500      72
1.8000       5
1.8500       7
1.9000       9
1.9500       2
2.0000      55
2.0500       5
2.1000       5
2.1500      12
2.2000       2
2.2500      56
2.3000       1
2.3500       9
2.4000       9
2.4500       1
2.5000      21
2.5500       1
2.6000       

In [4]:
compulsory_cols = [
    "stunt_difficulty",
    "stunt_dod",
    "stunt_max",
    "toss_difficulty",
    "standing_tumbling_difficulty",
    "standing_tumbling_dod",
    "running_tumbling_difficulty",
    "running_tumbling_dod",
    "running_tumbling_max",
    "jump_difficulty",
]

hit_df = analysis_df[
    analysis_df["is_hit"]
].copy()

ceiling_summary = []

for col in compulsory_cols:
    observed_max = hit_df[col].max()

    at_max = (
        hit_df[col] == observed_max
    ).sum()

    ceiling_summary.append({
        "category": col,
        "observed_max": observed_max,
        "median": hit_df[col].median(),
        "at_max": at_max,
        "pct_at_max": at_max / len(hit_df) * 100,
        "std_dev": hit_df[col].std(),
    })

ceiling_summary = (
    pd.DataFrame(ceiling_summary)
    .sort_values(
        "pct_at_max",
        ascending=False,
    )
)

print(
    ceiling_summary.to_string(
        index=False
    )
)

                    category  observed_max  median  at_max  pct_at_max  std_dev
 running_tumbling_difficulty        3.0000  3.0000    3369    100.0000   0.0000
standing_tumbling_difficulty        3.0000  3.0000    3368     99.9703   0.0086
        running_tumbling_dod        0.5000  0.5000    3365     99.8813   0.0131
             jump_difficulty        2.0000  2.0000    3360     99.7329   0.0394
             toss_difficulty        2.0000  2.0000    3359     99.7032   0.0502
            stunt_difficulty        4.5000  4.5000    3355     99.5844   0.0551
        running_tumbling_max        0.5000  0.5000    3343     99.2283   0.0343
                   stunt_max        0.7000  0.7000    3340     99.1392   0.0185
       standing_tumbling_dod        1.0000  1.0000    3334     98.9611   0.0397
                   stunt_dod        0.8000  0.8000    3056     90.7094   0.0533


In [6]:
zero_deduction_df = analysis_df[
    analysis_df["deductions"] == 0
].copy()

print(f"Zero-deduction performances: {len(zero_deduction_df):,}")

Zero-deduction performances: 3,369


In [7]:
subjective_difficulty_cols = [
    "pyramid_difficulty",
    "dance_difficulty",
]

subjective_summary = []

for col in subjective_difficulty_cols:
    observed_max = zero_deduction_df[col].max()

    at_max = (
        zero_deduction_df[col] == observed_max
    ).sum()

    subjective_summary.append({
        "category": col,
        "observed_max": observed_max,
        "min": zero_deduction_df[col].min(),
        "median": zero_deduction_df[col].median(),
        "at_max": at_max,
        "pct_at_max": at_max / len(zero_deduction_df) * 100,
        "std_dev": zero_deduction_df[col].std(),
        "unique_scores": zero_deduction_df[col].nunique(),
    })

subjective_summary = pd.DataFrame(
    subjective_summary
)

print(
    subjective_summary.to_string(
        index=False
    )
)

          category  observed_max    min  median  at_max  pct_at_max  std_dev  unique_scores
pyramid_difficulty        4.0000 3.2000  3.8000     374     11.1012   0.1073              7
  dance_difficulty        1.0000 0.5000  0.9000    1151     34.1644   0.0979              6


In [8]:
execution_cols = [
    "stunt_execution",
    "pyramid_execution",
    "toss_execution",
    "standing_tumbling_execution",
    "running_tumbling_execution",
    "jump_execution",
    "rc",
    "formations_transitions",
    "dance_execution",
    "show",
]

execution_summary = (
    zero_deduction_df[execution_cols]
    .agg(["min", "median", "max", "mean", "std"])
    .T
    .sort_values("std", ascending=False)
)

print(execution_summary.to_string())

                               min  median    max   mean    std
stunt_execution             3.2000  3.8000 4.0000 3.8210 0.1200
running_tumbling_execution  3.3000  3.8000 4.0000 3.8318 0.1196
pyramid_execution           3.2000  3.8000 4.0000 3.8265 0.1195
standing_tumbling_execution 3.4000  3.9000 4.0000 3.8542 0.1122
jump_execution              1.5000  1.9000 2.0000 1.8521 0.0966
toss_execution              1.3000  1.9000 2.0000 1.9118 0.0965
dance_execution             0.5000  0.8000 1.0000 0.8333 0.0934
formations_transitions      1.4000  2.0000 2.0000 1.9502 0.0806
show                        0.0000  1.8000 2.0000 1.8078 0.0802
rc                          1.5700  1.8300 2.0000 1.8160 0.0754
